# Data Information

This dataset contains medical insurance cost information for 1338 individuals. It includes demographic and health-related variables such as age, sex, BMI, number of children, smoking status, and residential region in the US. The target variable is charges, which represents the medical insurance cost billed to the individual.

Source: https://doi.org/10.34740/kaggle/dsv/12853160

# Dataset

In [202]:
import pandas as pd

data_df = pd.read_csv("insurance.csv")
data_df.head(10)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


# Problem Definition

## Objective

- use the dataset to estimate insurance bill/cost per person
- identify which variables are most associated with higher or lower insurance bill

## Usecase

Corporate Wellness Program
- Identify high-risk employees based on predicted healthcare costs
- Enable targeted interventions to reduce the company’s overall healthcare expenses

Insurance Company
- Analyze healthcare cost trends across different population segments
- Enable dynamic premium pricing based on individual risk profiles
- Provide personalized insurance plans tailored to each customer

Example Use Case
A person with certain conditions who is predicted to have high healthcare costs can be recommended lifestyle interventions

# Data Preparation

## 1. Data Understanding

In [203]:
print("Shape:", data_df.shape)

Shape: (1338, 7)


In [204]:
print("Columns:", data_df.columns)

Columns: Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='str')


In [205]:
data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [206]:
data_df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [207]:
data_df.duplicated().sum()

np.int64(1)

In [208]:
data_df = data_df.drop_duplicates()
data_df.duplicated().sum()

np.int64(0)

In [209]:
data_df.describe().round(2)

,age,bmi,children,charges
count,1337.00,1337.00,1337.00,1337.00
mean,39.22,30.66,1.10,13279.12
std,14.04,6.10,1.21,12110.36
min,18.00,15.96,0.00,1121.87
25%,27.00,26.29,0.00,4746.34
50%,39.00,30.40,1.00,9386.16
75%,51.00,34.70,2.00,16657.72
max,64.00,53.13,5.00,63770.43


In [210]:
data_df.describe(include="str")

,sex,smoker,region
count,1337,1337,1337
unique,2,2,4
top,male,no,southeast
freq,675,1063,364


# 2. Data Cleaning

## 2.1 Outlier 

In [211]:
num_cols = ['age', 'bmi']

for col in num_cols:
    Q1 = data_df[col].quantile(0.25)
    Q3 = data_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = data_df[(data_df[col] < lower) | (data_df[col] > upper)]
    
    print(f"{col}: {len(outliers)} outliers")

age: 0 outliers
bmi: 9 outliers


In [212]:
data_df['children'].value_counts()

children
0    573
1    324
2    240
3    157
4     25
5     18
Name: count, dtype: int64

In [213]:
data_df['region'].value_counts()

region
southeast    364
southwest    325
northwest    324
northeast    324
Name: count, dtype: int64

# 3. Data Transformation

## 3.1 Feature Encoding

In [214]:
data_df['Gender_Encoded'] = data_df['sex'].map({'male': 0, 'female': 1})
data_df.head(5)

,age,sex,bmi,children,smoker,region,charges,Gender_Encoded
0,19,female,27.900,0,yes,southwest,16884.92400,1
1,18,male,33.770,1,no,southeast,1725.55230,0
2,28,male,33.000,3,no,southeast,4449.46200,0
3,33,male,22.705,0,no,northwest,21984.47061,0
4,32,male,28.880,0,no,northwest,3866.85520,0


In [216]:
data_df['Smoker_Encoded'] = data_df['smoker'].map({'no': 0, 'yes': 1})
data_df.head(5)

,age,sex,bmi,children,smoker,region,charges,Gender_Encoded,Smoker_Encoded
0,19,female,27.900,0,yes,southwest,16884.92400,1,1
1,18,male,33.770,1,no,southeast,1725.55230,0,0
2,28,male,33.000,3,no,southeast,4449.46200,0,0
3,33,male,22.705,0,no,northwest,21984.47061,0,0
4,32,male,28.880,0,no,northwest,3866.85520,0,0


In [217]:
data_df['Region_Encoded'] = data_df['region'].map({'southeast': 0, 'southwest': 1, 'northeast': 2, 'northwest': 3})
data_df.head(5)

,age,sex,bmi,children,smoker,region,charges,Gender_Encoded,Smoker_Encoded,Region_Encoded
0,19,female,27.900,0,yes,southwest,16884.92400,1,1,1
1,18,male,33.770,1,no,southeast,1725.55230,0,0,0
2,28,male,33.000,3,no,southeast,4449.46200,0,0,0
3,33,male,22.705,0,no,northwest,21984.47061,0,0,3
4,32,male,28.880,0,no,northwest,3866.85520,0,0,3
